# Phase 7 Extension: YOLOv12 Medium Model Scaling

This notebook scales the YOLO architecture from Nano to Medium and extends training to 200 epochs to break the 66% recall ceiling. It trains purely on the Single-Class Defect Dataset on Google Drive.

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Install dependencies
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 8.6 MB/s eta 0:00:00


In [4]:
import os
import sys
import torch

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")

# Verify GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(torch.cuda.get_device_name(0))

Current working directory: /content/drive/MyDrive/sem_defect_project
Using device: cuda
Tesla T4


In [5]:
# Verify Dataset Paths
data_yaml_path = 'dataset_yolo_single_class/data.yaml'

if not os.path.exists(data_yaml_path):
    print(f"Error: Could not find {data_yaml_path}! Please ensure the dataset_yolo_single_class folder is at the root of sem_defect_project.")
else:
    print(f"Dataset config found at {data_yaml_path}")
    
import yaml
with open(data_yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)
    print("Classes:", cfg.get('names'))

Dataset config found at dataset_yolo_single_class/data.yaml
Classes: ['Defect']


## Step 1: Sanity Test (2 Epochs)
Verifies that the Medium model can load, compile, and execute properly without OOM errors before launching the 200 epoch run.

## Step 2: Full EXP-10 Training (200 Epochs)
Executes the optimized training routine.

In [7]:
print("Initializing YOLOv12m (Medium) for Full Training...")
try:
    model_full = YOLO('yolov12m.pt')
except Exception:
    model_full = YOLO('yolo11m.pt')

results_full = model_full.train(
    data=data_yaml_path,
    epochs=200,
    patience=50, # Generous early stopping
    imgsz=640,
    batch=16, # Keep at 16 to be safe with Medium model on a T4 GPU
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.01,
    project='runs/detect',
    name='EXP-10-YOLOv12m-640-200'
)

print("\n✅ EXP-10 Full Training Complete!")
print("Best weights saved at: runs/detect/EXP-10-YOLOv12m-640-200/weights/best.pt")


Initializing YOLOv12m (Medium) for Full Training...
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolo_single_class/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_sca